
# 練習問題: 2D 波動方程式 (膜の振動) を陽解法で解く

## 目標

太鼓の膜のような 2次元の波 `u_tt = c²(u_xx + u_yy)` を, 差分法 (陽解法) で時間発展させる。
空間の二重ループを `collapse(2)` で並列化する一方, 時間方向は前後のステップに依存するので逐次で進める, という時間発展シミュレーションの構造を学ぶ。

## 課題

`wave.cpp` (または `wave.f90`) は, L×L の膜の四辺を固定 (u=0) し, 中央にガウスの山を初期条件として置く。各時間ステップで, 内部の各点を

```
u^{n+1} = 2 u^n − u^{n-1} + coef × (上 + 下 + 左 + 右 − 4×中央)
```

で更新する (5点ラプラシアン)。`coef = (c·Δt/Δx)²` で, 安定条件は `coef ≤ 0.5` (このコードでは 0.25)。

`TODO` の箇所に **OpenMP の指示を追加** し, 内部点を更新する空間の二重ループを並列化せよ。

- C++: 二重ループの直前に `#pragma omp parallel for collapse(2)` を1行加える。
- Fortran: 二重ループを `!$omp parallel do collapse(2) private(lap)` と `!$omp end parallel do` で囲む。

ポイント:

- 並列化するのは**空間ループ** (各時間ステップ内)。**時間ループは並列化できない** (u^{n+1} が u^n, u^{n-1} に依存するため)。
- 過去・現在・未来の3枚の配列をポインタで回して使う (B1 と同様, コピーしない)。
- 各点の更新は独立で総和も無いので reduction は不要 (B1 の残差判定とはそこが違う)。

## コンパイルと実行

```
# C++
nvc++ -fast -mp=multicore wave.cpp -o wave.exe

# Fortran
nvfortran -fast -mp=multicore wave.f90 -o wave.exe
```

第1引数で格子サイズ `L`, 第2引数で時間ステップ数。

```
OMP_NUM_THREADS=4 ./wave.exe 257 200
```

## 期待される結果

```
L=257, steps=200: 最大振幅=0.1424, 対称性誤差=7.22e-16 (≈0 なら正しい)
```

- **安定性**: 振幅は O(1) のまま (発散しない)。ステップ数を増やしても爆発しなければ正しい (試しに `coef` を 0.6 に増やすと発散することも見られる)。
- **対称性による検算**: 初期条件が `i ↔ j` 対称なので, 解も常に対称になるはず。`max|u[i][j] − u[j][i]|` が丸め誤差 (1e-15 程度) ならステンシルが正しく実装できている。
- スレッド数を変えても結果は変わらない (各点の更新が独立な決定的計算なので)。
- 余裕があれば途中のスナップショットを出力し, 波が広がって四辺で反射する様子を観察せよ。



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ wave.cpp
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <omp.h>

/* 2D 波動方程式 u_tt = c^2 (u_xx + u_yy) を陽解法で時間発展させる。
   L×L の膜。四辺を固定 (u=0) し, 中央に山(ガウス)を置いて波が広がり反射する様子を計算する。
   更新式 (5点ラプラシアン, coef = (c*dt/dx)^2, 安定条件 coef <= 0.5):
     u^{n+1} = 2 u^n - u^{n-1} + coef * (上+下+左+右 - 4*中央)
   時間方向は前後のステップに依存するので逐次, 空間の二重ループを並列化する。 */
int main(int argc, char ** argv) {
  int    L     = (argc > 1 ? atoi(argv[1]) : 257);     /* 一辺の格子点数 */
  int    steps = (argc > 2 ? atoi(argv[2]) : 200);     /* 時間ステップ数 */
  double coef  = 0.25;                                 /* (c*dt/dx)^2, 安定 */

  int n = L * L;
  double * up = (double *)calloc(n, sizeof(double));   /* u^{n-1} */
  double * cu = (double *)calloc(n, sizeof(double));   /* u^{n}   */
  double * nx = (double *)calloc(n, sizeof(double));   /* u^{n+1} */
  /* 初期条件: 中央にガウスの山, 初速 0 (= up と cu を同じにする)。境界は 0 のまま。 */
  double c0 = (L - 1) / 2.0, sig = L / 16.0;
  for (int i = 1; i < L - 1; i++)
    for (int j = 1; j < L - 1; j++) {
      double r2 = (i - c0) * (i - c0) + (j - c0) * (j - c0);
      double v = exp(-r2 / (2.0 * sig * sig));
      up[i * L + j] = v;
      cu[i * L + j] = v;
    }

  double t0 = omp_get_wtime();
  for (int t = 0; t < steps; t++) {
    /* 内部の各点を更新 (時間1ステップ進める) */
    // TODO: 内側の二重ループを #pragma omp parallel for collapse(2) で並列化せよ.
    for (int i = 1; i < L - 1; i++) {
      for (int j = 1; j < L - 1; j++) {
        double lap = cu[(i - 1) * L + j] + cu[(i + 1) * L + j]
                   + cu[i * L + (j - 1)] + cu[i * L + (j + 1)]
                   - 4.0 * cu[i * L + j];
        nx[i * L + j] = 2.0 * cu[i * L + j] - up[i * L + j] + coef * lap;
      }
    }
    /* up <- cu <- nx と時間を1つ進める (ポインタを回す) */
    double * tmp = up; up = cu; cu = nx; nx = tmp;
  }
  double elapsed = omp_get_wtime() - t0;

  /* 検算1: 最大振幅 (発散していなければ O(1) のまま)。
     検算2: 初期条件が i<->j 対称なので, 解も常に対称。max|u[i][j]-u[j][i]| は丸め誤差程度。 */
  double maxabs = 0.0, asym = 0.0;
  for (int i = 0; i < L; i++)
    for (int j = 0; j < L; j++) {
      double a = fabs(cu[i * L + j]);
      if (a > maxabs) maxabs = a;
      double s = fabs(cu[i * L + j] - cu[j * L + i]);
      if (s > asym) asym = s;
    }
  printf("L=%d, steps=%d: 最大振幅=%.4f, 対称性誤差=%.2e (≈0 なら正しい)\n",
         L, steps, maxabs, asym);
  printf("elapsed = %.3f sec\n", elapsed);
  free(up); free(cu); free(nx);
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore wave.cpp -o wave_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./wave_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ wave.f90
! 2D 波動方程式 u_tt = c^2 (u_xx + u_yy) を陽解法で時間発展させる。
! L×L の膜。四辺を固定 (u=0) し, 中央に山(ガウス)を置いて波が広がり反射する様子を計算する。
! 更新式 (5点ラプラシアン, coef=(c*dt/dx)^2, 安定条件 coef<=0.5):
!   u^{n+1} = 2 u^n - u^{n-1} + coef*(上+下+左+右 - 4*中央)
! 時間方向は前後ステップに依存するので逐次, 空間の二重ループを並列化する。
program wave
  use omp_lib
  implicit none
  integer :: L, steps, t, i, j
  real(8) :: coef, c0, sig, r2, v, lap, maxabs, asym, s, a, t0, elapsed
  real(8), allocatable, target :: arr1(:,:), arr2(:,:), arr3(:,:)
  real(8), pointer :: up(:,:), cu(:,:), nx(:,:), tmp(:,:)
  character(len=32) :: arg
  L = 257; steps = 200; coef = 0.25d0
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read (arg, *) L
  end if
  if (command_argument_count() >= 2) then
     call get_command_argument(2, arg); read (arg, *) steps
  end if

  allocate(arr1(0:L-1,0:L-1), arr2(0:L-1,0:L-1), arr3(0:L-1,0:L-1))
  arr1 = 0.0d0; arr2 = 0.0d0; arr3 = 0.0d0
  up => arr1; cu => arr2; nx => arr3
  ! 初期条件: 中央にガウスの山, 初速 0 (up と cu を同じに)。境界は 0 のまま。
  c0 = (L - 1) / 2.0d0; sig = L / 16.0d0
  do j = 1, L - 2
     do i = 1, L - 2
        r2 = (i - c0)**2 + (j - c0)**2
        v = exp(-r2 / (2.0d0 * sig * sig))
        up(i,j) = v
        cu(i,j) = v
     end do
  end do

  t0 = omp_get_wtime()
  do t = 1, steps
     ! 内部の各点を更新 (時間1ステップ進める)
     ! TODO: 内側の二重ループを !$omp parallel do collapse(2) private(lap) で並列化せよ.
     do j = 1, L - 2
        do i = 1, L - 2
           lap = cu(i-1,j) + cu(i+1,j) + cu(i,j-1) + cu(i,j+1) - 4.0d0 * cu(i,j)
           nx(i,j) = 2.0d0 * cu(i,j) - up(i,j) + coef * lap
        end do
     end do
     ! TODO: 上で始めた parallel do 領域を閉じる (!$omp end parallel do).
     ! up <- cu <- nx と時間を1つ進める (ポインタを回す)
     tmp => up; up => cu; cu => nx; nx => tmp
  end do
  elapsed = omp_get_wtime() - t0

  ! 検算1: 最大振幅。検算2: i<->j 対称性の誤差 (≈0 なら正しい)。
  maxabs = 0.0d0; asym = 0.0d0
  do j = 0, L - 1
     do i = 0, L - 1
        a = abs(cu(i,j))
        if (a > maxabs) maxabs = a
        s = abs(cu(i,j) - cu(j,i))
        if (s > asym) asym = s
     end do
  end do
  print "(a,i0,a,i0,a,f0.4,a,es9.2,a)", &
       "L=", L, ", steps=", steps, ": 最大振幅=", maxabs, ", 対称性誤差=", asym, " (≈0 なら正しい)"
  print "(a,f0.3,a)", "elapsed = ", elapsed, " sec"
end program wave

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore wave.f90 -o wave_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./wave_f90.exe


# 4. 発展目標 (できる範囲で挑戦)
- この問題の基本は **マルチコア並列化** (`#pragma omp parallel for` / `!$omp parallel do` など)。まずはここまでを目指す。
- 余力があれば次にも挑戦:
  - **GPUで並列化**: コンパイルを `-mp=gpu` にして, 重いループに `#pragma omp target teams distribute parallel for` (+ 必要に応じて `map`) を付ける。
  - **SIMD化** (11/12章): 内側ループに `#pragma omp simd`, またはベクトル型。 **ILP向上** (13章): ベクトル長 `nl` の調整。
- どこまで速くできるか挑戦し, その効果を下の「性能を比べる」で可視化しよう。

# 5. 性能を比べる (任意)
- 各プログラムは主計算部分の所要時間を `elapsed = ... sec` の形で表示する。
- 構成を変えて (スレッド数, SIMDの有無, GPU など) 実行し, 得られた **経過秒** を下の `DATA` に「ラベルと秒」で並べて実行すると, 棒グラフと「基準(先頭)比のスピードアップ」が出る。
- 試した構成だけ書けばよい (`#` で始まる行は無視)。


In [ ]:
import matplotlib.pyplot as plt

# 各構成の (ラベル, 経過秒) を並べる。先頭の行を基準(スピードアップ=1)にする。
# 自分が実際に試した構成の数値に書き換える。
DATA = [
    ("serial",    1.00),
    ("omp-8",     0.20),
    # ("simd",      0.50),
    # ("simd+omp",  0.07),
    # ("gpu",       0.05),
]

labels = [d[0] for d in DATA]
secs   = [float(d[1]) for d in DATA]
speed  = [secs[0] / s for s in secs]                 # 先頭(基準)比のスピードアップ
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].bar(labels, secs)
ax[0].set_ylabel("elapsed (s)")
ax[0].set_title("time (lower = faster)")
ax[1].bar(labels, speed)
ax[1].set_ylabel("speedup vs " + labels[0])
ax[1].set_title("speedup (higher = faster)")
for a in ax:
    a.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


# 6. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:wave.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 6-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 6-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 6-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:wave.cpp}

コマンドと実行結果:
{bash[-1]}

## 6-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:wave.cpp}